In [1]:
%%time

!python ../scripts/analyze_chronos_results.py --results-dir ../results/baselines --output-dir ../results/baselines

CHRONOS BASELINE RESULTS ANALYSIS
Results directory: ../results/baselines
Output directory: ../results/baselines
Loaded 45 rows from ../results/baselines/zero-shot_chronos-2_all_stocks_results.csv
Loaded 45 rows from ../results/baselines/multivariate_chronos-2_all_stocks_results.csv
Loaded 45 rows from ../results/baselines/finetuned_single/Chronos2_finetuned_all_stocks_results.csv
Loaded 45 rows from ../results/baselines/finetuned_covariates/Chronos2_finetuned_cov_all_stocks_results.csv

Loaded 180 result rows
Stocks: ['AAPL', 'META', 'NVDA', 'SPY', 'TSLA']
Strategies: ['long_short']
Model Types: ['zero-shot', 'multivariate', 'finetuned', 'finetuned_cov']
Horizons: [2, 3, 4, 5, 6, 7, 8, 9, 10]

ANALYSIS BY STRATEGY

--- LONG_SHORT ---
  Samples: 180
  Avg Accuracy:    0.4911 (+/- 0.0410)
  Avg ROC-AUC:     0.4791 (+/- 0.0497)
  Avg Trades:      53.2
  Avg Win Rate:    0.4409 (44.1%)
  Avg Sharpe:      -0.7140
  Avg Total Return: -0.0781 (-7.81%)

ANALYSIS BY STOCK

--- AAPL ---
  Sampl

In [2]:
# ============================================================================
# FinCast Foundation Model Baseline
# ============================================================================

import pandas as pd

# Load FinCast results
fincast_df = pd.read_csv("../results/baselines/fincast_all_stocks_results.csv")

print("FinCast Baseline Results")
print("=" * 80)
print(f"Total records: {len(fincast_df)}")
print(f"Stocks: {fincast_df['Stock'].unique().tolist()}")
print(f"Horizons: {sorted(fincast_df['Horizon'].unique().tolist())}")

# Summary statistics
print("\n" + "=" * 80)
print("Overall Summary (averaged across all stocks and horizons)")
print("=" * 80)
print(f"Accuracy: {fincast_df['Test_Accuracy'].mean():.3f}")
print(f"AUC: {fincast_df['Test_ROC_AUC'].mean():.3f}")
print(f"Sharpe: {fincast_df['Sharpe'].mean():.2f}")
print(f"Win Rate: {fincast_df['WinRate'].mean()*100:.1f}%")
print(f"Total Return: {fincast_df['TotalReturn'].mean()*100:.1f}%")

# Per-stock summary
print("\n" + "=" * 80)
print("Per-Stock Summary")
print("=" * 80)
stock_summary = fincast_df.groupby('Stock').agg({
 'Test_Accuracy': 'mean',
 'Test_ROC_AUC': 'mean',
 'Sharpe': 'mean',
 'WinRate': 'mean',
 'TotalReturn': 'mean'
}).round(3)
stock_summary.columns = ['Accuracy', 'AUC', 'Sharpe', 'WinRate', 'Return']
stock_summary['WinRate'] = (stock_summary['WinRate'] * 100).round(1)
stock_summary['Return'] = (stock_summary['Return'] * 100).round(1)
print(stock_summary.to_string())

# Best by AUC and Best by Sharpe (per stock, then average)
STOCKS = ["AAPL", "META", "NVDA", "SPY", "TSLA"]

def get_fincast_best_by_metric(df, metric_col):
    """Get best config per stock by given metric."""
    best_rows = []
    for stock in STOCKS:
        stock_data = df[df['Stock'] == stock]
        if len(stock_data) > 0:
            best_idx = stock_data[metric_col].idxmax()
            best_rows.append(stock_data.loc[best_idx])
    if best_rows:
        best_df = pd.DataFrame(best_rows)
        return {
            'Acc': best_df['Test_Accuracy'].mean(),
            'AUC': best_df['Test_ROC_AUC'].mean(),
            'N': best_df['Trades'].mean(),
            'WinRate': best_df['WinRate'].mean() * 100,
            'Sharpe': best_df['Sharpe'].mean()
        }
    return None

print("\n" + "=" * 80)
print("FOR TABLE VIII: FinCast Baseline Comparison")
print("=" * 80)
print("Method: Pick best config per stock, then average across 5 stocks\n")

print(f"{'Method':<45} {'Acc':>6} {'AUC':>6} {'N':>5} {'Win%':>6} {'Sharpe':>7} ")
print("-" * 80)

fc_auc = get_fincast_best_by_metric(fincast_df, 'Test_ROC_AUC')
fc_sharpe = get_fincast_best_by_metric(fincast_df, 'Sharpe')

if fc_auc:
    print(f"{'FinCast (Best by AUC)':<45} {fc_auc['Acc']:>6.3f} {fc_auc['AUC']:>6.3f} {fc_auc['N']:>5.1f} {fc_auc['WinRate']:>6.1f} {fc_auc['Sharpe']:>7.2f} ")
if fc_sharpe:
    print(f"{'FinCast (Best by Sharpe)':<45} {fc_sharpe['Acc']:>6.3f} {fc_sharpe['AUC']:>6.3f} {fc_sharpe['N']:>5.1f} {fc_sharpe['WinRate']:>6.1f} {fc_sharpe['Sharpe']:>7.2f} ")

print("-" * 80)

# LaTeX table rows
print("\nLaTeX Table Rows:")
print("-" * 80)
if fc_auc:
    print(f"FinCast (Best by AUC) & {fc_auc['Acc']:.3f} & {fc_auc['AUC']:.3f} & {fc_auc['Sharpe']:.2f} & {fc_auc['WinRate']:.1f} \\\\")
if fc_sharpe:
    print(f"FinCast (Best by Sharpe) & {fc_sharpe['Acc']:.3f} & {fc_sharpe['AUC']:.3f} & {fc_sharpe['Sharpe']:.2f} & {fc_sharpe['WinRate']:.1f} \\\\")

FinCast Baseline Results
Total records: 45
Stocks: ['AAPL', 'META', 'NVDA', 'SPY', 'TSLA']
Horizons: [2, 3, 4, 5, 6, 7, 8, 9, 10]

Overall Summary (averaged across all stocks and horizons)
Accuracy: 0.505
AUC: 0.510
Sharpe: -1.11
Win Rate: 41.0%
Total Return: -29.0%

Per-Stock Summary
       Accuracy    AUC  Sharpe  WinRate  Return
Stock                                          
AAPL      0.503  0.482  -0.577     46.0   -12.0
META      0.485  0.511  -1.496     39.0   -41.1
NVDA      0.480  0.517  -1.081     40.1   -47.9
SPY       0.532  0.526  -2.174     34.6   -22.7
TSLA      0.525  0.515  -0.208     45.5   -21.4

FOR TABLE VIII: FinCast Baseline Comparison
Method: Pick best config per stock, then average across 5 stocks

Method                                           Acc    AUC     N   Win%  Sharpe 
--------------------------------------------------------------------------------
FinCast (Best by AUC)                          0.524  0.560  76.0   48.1   -0.77 
FinCast (Best by Sharp

In [3]:
# ============================================================================
# Combined Baseline Comparison (Including FinCast Zero-Shot and Finetuned)
# ============================================================================
import pandas as pd

STOCKS = ["AAPL", "META", "NVDA", "SPY", "TSLA"]

# Load all baseline results
chronos_zs_df = pd.read_csv("../results/baselines/zero-shot_chronos-2_all_stocks_results.csv")
chronos_zs_df = chronos_zs_df[chronos_zs_df['Strategy'] == 'long_short']

chronos_mv_df = pd.read_csv("../results/baselines/multivariate_chronos-2_all_stocks_results.csv")
chronos_mv_df = chronos_mv_df[chronos_mv_df['Strategy'] == 'long_short']

finetuned_df = pd.read_csv("../results/baselines/finetuned_single/Chronos2_finetuned_all_stocks_results.csv")
finetuned_df = finetuned_df[finetuned_df['Strategy'] == 'long_short']

# finetuned_cov_df = pd.read_csv("finetuned_covariates/Chronos2_finetuned_cov_all_stocks_inference_results.csv")
finetuned_cov_df = pd.read_csv("../results/baselines/finetuned_covariates/Chronos2_finetuned_cov_all_stocks_results.csv")
finetuned_cov_df = finetuned_cov_df[finetuned_cov_df['Strategy'] == 'long_short']

fincast_df = pd.read_csv("../results/baselines/fincast_all_stocks_results.csv")

# Load FinCast Finetuned results
fincast_finetuned_df = pd.read_csv("../results/baselines/finetuned_fincast/fincast_finetuned_all_stocks_results.csv")

def get_best_by_metric(df, metric_col, acc_col='Test_Accuracy', auc_col='Test_ROC_AUC'):
    """Get best config per stock by given metric, return averaged results."""
    best_rows = []
    for stock in STOCKS:
        stock_data = df[df['Stock'] == stock]
        if len(stock_data) > 0:
            best_idx = stock_data[metric_col].idxmax()
            best_rows.append(stock_data.loc[best_idx])
    if best_rows:
        best_df = pd.DataFrame(best_rows)
        return {
            'Acc': best_df[acc_col].mean() if acc_col in best_df.columns else best_df['Accuracy'].mean(),
            'AUC': best_df[auc_col].mean() if auc_col in best_df.columns else 0.5,
            'N': best_df['Trades'].mean(),
            'WinRate': best_df['WinRate'].mean() * 100,
            'Sharpe': best_df['Sharpe'].mean()
        }
    return None


print("=" * 100)
print("COMPLETE BASELINE COMPARISON (Including FinCast Zero-Shot and Finetuned)")
print("=" * 100)
print("Method: Pick best config per stock, then average across 5 stocks\n")

print(f"{'Method':<50} {'Acc':>6} {'AUC':>6} {'N':>5} {'Win%':>6} {'Sharpe':>7} ")
print("=" * 100)


print("-" * 100)

# FinCast zero-shot
print("FinCast (zero-shot):")
fc_auc = get_best_by_metric(fincast_df, 'Test_ROC_AUC')
fc_sharpe = get_best_by_metric(fincast_df, 'Sharpe')
if fc_auc:
    print(f"{' Best by AUC':<50} {fc_auc['Acc']:>6.3f} {fc_auc['AUC']:>6.3f} {fc_auc['N']:>5.1f} {fc_auc['WinRate']:>6.1f} {fc_auc['Sharpe']:>7.2f} ")
if fc_sharpe:
    print(f"{' Best by Sharpe':<50} {fc_sharpe['Acc']:>6.3f} {fc_sharpe['AUC']:>6.3f} {fc_sharpe['N']:>5.1f} {fc_sharpe['WinRate']:>6.1f} {fc_sharpe['Sharpe']:>7.2f} ")

print("-" * 100)

# FinCast finetuned (NEW)
print("FinCast (fine-tuned):")
fcft_auc = get_best_by_metric(fincast_finetuned_df, 'Test_ROC_AUC')
fcft_sharpe = get_best_by_metric(fincast_finetuned_df, 'Sharpe')
if fcft_auc:
    print(f"{' Best by AUC':<50} {fcft_auc['Acc']:>6.3f} {fcft_auc['AUC']:>6.3f} {fcft_auc['N']:>5.1f} {fcft_auc['WinRate']:>6.1f} {fcft_auc['Sharpe']:>7.2f} ")
if fcft_sharpe:
    print(f"{' Best by Sharpe':<50} {fcft_sharpe['Acc']:>6.3f} {fcft_sharpe['AUC']:>6.3f} {fcft_sharpe['N']:>5.1f} {fcft_sharpe['WinRate']:>6.1f} {fcft_sharpe['Sharpe']:>7.2f} ")

print("-" * 100)

# Chronos-2 zero-shot
print("Chronos-2 (zero-shot):")
zs_auc = get_best_by_metric(chronos_zs_df, 'Test_ROC_AUC')
zs_sharpe = get_best_by_metric(chronos_zs_df, 'Sharpe')
if zs_auc:
    print(f"{' Best by AUC':<50} {zs_auc['Acc']:>6.3f} {zs_auc['AUC']:>6.3f} {zs_auc['N']:>5.1f} {zs_auc['WinRate']:>6.1f} {zs_auc['Sharpe']:>7.2f} ")
if zs_sharpe:
    print(f"{' Best by Sharpe':<50} {zs_sharpe['Acc']:>6.3f} {zs_sharpe['AUC']:>6.3f} {zs_sharpe['N']:>5.1f} {zs_sharpe['WinRate']:>6.1f} {zs_sharpe['Sharpe']:>7.2f} ")

print("-" * 100)

# Chronos-2 + Sentiment zero-shot
print("Chronos-2 + Sentiment (zero-shot):")
mv_auc = get_best_by_metric(chronos_mv_df, 'Test_ROC_AUC')
mv_sharpe = get_best_by_metric(chronos_mv_df, 'Sharpe')
if mv_auc:
    print(f"{' Best by AUC':<50} {mv_auc['Acc']:>6.3f} {mv_auc['AUC']:>6.3f} {mv_auc['N']:>5.1f} {mv_auc['WinRate']:>6.1f} {mv_auc['Sharpe']:>7.2f} ")
if mv_sharpe:
    print(f"{' Best by Sharpe':<50} {mv_sharpe['Acc']:>6.3f} {mv_sharpe['AUC']:>6.3f} {mv_sharpe['N']:>5.1f} {mv_sharpe['WinRate']:>6.1f} {mv_sharpe['Sharpe']:>7.2f} ")

print("-" * 100)

# Chronos-2 fine-tuned
print("Chronos-2 (fine-tuned):")
ft_auc = get_best_by_metric(finetuned_df, 'Test_ROC_AUC')
ft_sharpe = get_best_by_metric(finetuned_df, 'Sharpe')
if ft_auc:
    print(f"{' Best by AUC':<50} {ft_auc['Acc']:>6.3f} {ft_auc['AUC']:>6.3f} {ft_auc['N']:>5.1f} {ft_auc['WinRate']:>6.1f} {ft_auc['Sharpe']:>7.2f} ")
if ft_sharpe:
    print(f"{' Best by Sharpe':<50} {ft_sharpe['Acc']:>6.3f} {ft_sharpe['AUC']:>6.3f} {ft_sharpe['N']:>5.1f} {ft_sharpe['WinRate']:>6.1f} {ft_sharpe['Sharpe']:>7.2f} ")

print("-" * 100)

# Chronos-2 + Sentiment fine-tuned
print("Chronos-2 + Sentiment (fine-tuned):")
ftc_auc = get_best_by_metric(finetuned_cov_df, 'Test_ROC_AUC')
ftc_sharpe = get_best_by_metric(finetuned_cov_df, 'Sharpe')
if ftc_auc:
    print(f"{' Best by AUC':<50} {ftc_auc['Acc']:>6.3f} {ftc_auc['AUC']:>6.3f} {ftc_auc['N']:>5.1f} {ftc_auc['WinRate']:>6.1f} {ftc_auc['Sharpe']:>7.2f} ")
if ftc_sharpe:
    print(f"{' Best by Sharpe':<50} {ftc_sharpe['Acc']:>6.3f} {ftc_sharpe['AUC']:>6.3f} {ftc_sharpe['N']:>5.1f} {ftc_sharpe['WinRate']:>6.1f} {ftc_sharpe['Sharpe']:>7.2f} ")

print("=" * 100)
print("\nNote: FinCast uses price-only context (no sentiment). All results averaged across 5 stocks.")

COMPLETE BASELINE COMPARISON (Including FinCast Zero-Shot and Finetuned)
Method: Pick best config per stock, then average across 5 stocks

Method                                                Acc    AUC     N   Win%  Sharpe 
----------------------------------------------------------------------------------------------------
FinCast (zero-shot):
 Best by AUC                                        0.524  0.560  76.0   48.1   -0.77 
 Best by Sharpe                                     0.474  0.507  59.6   45.8   -0.10 
----------------------------------------------------------------------------------------------------
FinCast (fine-tuned):
 Best by AUC                                        0.596  0.693  13.6   39.2   -0.40 
 Best by Sharpe                                     0.546  0.529  21.8   44.6    0.16 
----------------------------------------------------------------------------------------------------
Chronos-2 (zero-shot):
 Best by AUC                                        0.520

In [5]:
!python ../scripts/update_sota_table.py --results-dir ../results/baselines --tex-file ../docs/main.tex

UPDATING SOTA TABLE
Results directory: ../results/baselines
LaTeX file: ../docs/main.tex

Loading Chronos-2 results...
Loaded 180 Chronos-2 result rows
Model types: ['zero-shot' 'multivariate' 'finetuned' 'finetuned_cov']
Stocks: ['AAPL' 'META' 'NVDA' 'SPY' 'TSLA']

Loading FinCast results...
  Loaded 45 zero-shot results
  Loaded 45 fine-tuned results
Loaded 90 FinCast result rows
Model types: ['zero-shot' 'finetuned']
Stocks: ['AAPL' 'META' 'NVDA' 'SPY' 'TSLA']

Calculating baseline metrics...

Baseline Metrics Summary:
--------------------------------------------------------------------------------

FinCast Baselines:

fincast_zeroshot:
  AUC-selected: Acc=0.524, AUC=0.560, Sharpe=-0.77
  Sharpe-selected: Acc=0.474, AUC=0.507, Sharpe=-0.10

fincast_finetuned:
  AUC-selected: Acc=0.596, AUC=0.693, Sharpe=-0.40
  Sharpe-selected: Acc=0.546, AUC=0.529, Sharpe=0.16

Chronos-2 Baselines:

chronos_zeroshot_price:
  AUC-selected: Acc=0.520, AUC=0.512, Sharpe=-0.47
  Sharpe-selected: Acc=0.